<a href="https://colab.research.google.com/github/Cato-8/Machine-Learning-Labs/blob/main/Lab8_SVM_Template_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB 08 — Support Vector Machines
## Kernels, Margins, and Model Selection on Real Customer-Churn Data

## Student Information
- **Name:Marvi Malik**
- **Student ID:023-24-0016**
- **Section:B**
- **GitHub:**
- **Kaggle:**
- **Dataset:** Telco Customer Churn (Kaggle)
- **Dataset Source:** <paste the Kaggle URL you used>

---

### Before You Start
- You already studied SVM theory in class (margin, support vectors, kernel trick, C, gamma). This notebook does **not** re-teach that theory.
- This is **not** a "load data → `SVC().fit()` → done" lab. Most marks come from experimentation, visualization and justification, not from the fact that a model trained.
- You already know Python, Pandas, Matplotlib, `train_test_split`, encoding and evaluation metrics from Labs 2-6.
- Each task gives an **objective** and a **question** — not a code recipe. Hints tell you which tool is relevant; you decide how to use it.
- Do **not** provide complete code for every question yourself either — work it out. Every plot needs a written interpretation directly below it.


---

## Part 0 — Recap and Setup

Reload the Telco Customer Churn dataset and reapply the **same** cleaning/encoding pipeline from Labs 2/3/4/6:
- Convert `TotalCharges` to numeric and handle missing values
- Drop `customerID`
- Map `Churn` to 0/1
- One-hot encode categorical columns (`drop_first=True`)
- `train_test_split` with `stratify=y` — use the **same `random_state`** you used in Labs 3/4/6 so results stay comparable


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                              precision_recall_curve)

df = pd.read_csv("clean_churn.csv")

# In case clean_churn.csv isn't pre-cleaned, re-apply Lab 2/3 cleaning defensively:
if df['TotalCharges'].dtype.kind not in 'if':
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
if df['Churn'].dtype.kind not in 'iuf':
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(df.shape)
df.head()



(7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [2]:
X = df.drop(columns=['Churn'])
y = df['Churn']

categorical_cols = X.select_dtypes(exclude=['number','bool']).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
X_train.dtypes.value_counts()



(5634, 30) (1409, 30)


,count
bool,26
int64,2
float64,2


---

## Part 1 — Why Scaling Matters for SVM

**Task 1.1 — Objective:** Demonstrate what happens to an SVM when features are not scaled.

**What to do:**
1. Train a linear-kernel SVM (`SVC(kernel='linear')`) directly on your **un-scaled** encoded features.
2. Train a second linear-kernel SVM on the same split after applying `StandardScaler` (fit on train only, transform on train and test).
3. Compare accuracy, F1-score, training time, and any convergence warnings between the two runs.

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# unscaled
svm_raw = SVC(kernel='linear', random_state=42)
svm_raw.fit(X_train, y_train)

# scaled  (fit_transform on TRAIN, transform ONLY on TEST)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
```


In [ ]:
start = time.time()
svm_raw = SVC(kernel='linear', random_state=42)
svm_raw.fit(X_train, y_train)
time_raw = time.time() - start

y_pred_raw = svm_raw.predict(X_test)
acc_raw = accuracy_score(y_test, y_pred_raw)
f1_raw = f1_score(y_test, y_pred_raw)
print("UNSCALED  Accuracy:", acc_raw, " F1:", f1_raw, " Time(s):", time_raw)


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

start = time.time()
svm_scaled = SVC(kernel='linear', random_state=42)
svm_scaled.fit(X_train_s, y_train)
time_scaled = time.time() - start

y_pred_scaled = svm_scaled.predict(X_test_s)
acc_scaled = accuracy_score(y_test, y_pred_scaled)
f1_scaled = f1_score(y_test, y_pred_scaled)
print("SCALED  Accuracy:", acc_scaled, " F1:", f1_scaled, " Time(s):", time_scaled)


**Questions to answer:**
(a) What differences did you observe between the unscaled and scaled models (accuracy, warnings, training time)?

*Your answer:* The difference in training time was dramatic — the unscaled linear SVM took about 376 seconds (over 6 minutes) to train, while the scaled version took under 1 second. Accuracy was fairly close (unscaled 0.794 vs scaled 0.789), but F1-score was clearly better on the scaled model (0.569 vs 0.526), meaning the scaled model does a better job balancing precision and recall on the minority (churn) class.

(b) Why does an unscaled linear SVM behave this way, in terms of how the margin/decision boundary is computed?

*Your answer:* SVM tries to find the maximum-margin hyperplane by solving an optimization problem that depends on distances between data points. When features are on very different scales (e.g. `TotalCharges` in the thousands vs `SeniorCitizen` as 0/1), features with larger numeric ranges dominate the distance/margin calculations, distorting the geometry the optimizer has to search through. This makes the underlying quadratic optimization problem much harder to converge on, which is why training took so much longer without affecting accuracy very much.

(c) From this point onward in the lab, which version will you use for every subsequent SVM, and why?

*Your answer:* I will use the scaled version for every subsequent SVM in this lab, since it trains dramatically faster (under 1 second vs over 6 minutes) and gives a better F1-score, with no real downside.


---

## Part 2 — Kernel Comparison

**Task 2.1 — Objective:** Train and fairly compare SVM classifiers using the linear, RBF and polynomial kernels on your **scaled** features.

**What to do:**
1. Train three `SVC` models — `kernel='linear'`, `kernel='rbf'`, `kernel='poly'` — with the same `random_state`, on the same scaled train split.
2. For each, record: Accuracy, Precision, Recall, F1-score, and number of support vectors (`model.support_.shape[0]`).
3. Assemble a single comparison table (a DataFrame is fine).
4. Plot the confusion matrix for whichever kernel performed best on F1-score.

| Syntax | Description |
|---|---|
| `SVC(kernel=...)` | `'linear'`, `'rbf'`, or `'poly'` |
| `model.support_` | indices of support vectors |
| `model.n_support_` | number of support vectors per class |


In [ ]:
kernels = ['linear', 'rbf', 'poly']
kernel_results = []

for k in kernels:
    m = SVC(kernel=k, random_state=42)
    m.fit(X_train_s, y_train)
    pred = m.predict(X_test_s)
    kernel_results.append({
        'Kernel': k,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred),
        'Support Vectors': m.support_.shape[0],
        'model': m,
        'preds': pred
    })

for r in kernel_results:
    print(r['Kernel'], r['Accuracy'], r['Precision'], r['Recall'], r['F1-score'], r['Support Vectors'])


In [ ]:
kernel_table = pd.DataFrame(kernel_results)[['Kernel','Accuracy','Precision','Recall','F1-score','Support Vectors']]
kernel_table


In [ ]:
best_kernel_row = max(kernel_results, key=lambda r: r['F1-score'])
print("Best kernel by F1:", best_kernel_row['Kernel'])

cm = confusion_matrix(y_test, best_kernel_row['preds'])
ConfusionMatrixDisplay(cm).plot()
plt.title(f"Confusion Matrix — {best_kernel_row['Kernel']} kernel")
plt.show()


**Questions to answer:**
(a) Which kernel produced the best F1-score on the test set? Does this surprise you given the theory covered in class?

*Your answer:* The linear kernel produced the best F1-score (0.569), slightly ahead of RBF (0.556) and poly (0.540), even though RBF had the highest accuracy (0.793) and precision (0.644). This is a bit surprising since RBF/poly are usually expected to capture more complex patterns, but it suggests the relationship between these churn features and the target is close to linearly separable, so the extra flexibility of RBF/poly doesn't pay off here — if anything it trades recall for precision.

(b) Which kernel used the most support vectors, and what does a high support-vector count suggest about the complexity of the decision boundary it learned?

*Your answer:* The poly kernel used the most support vectors (2736), followed by RBF (2657) and linear (2567). A higher support-vector count suggests a more complex, less generalizable decision boundary — the model needs more points to define its boundary, which can be a sign it is fitting more intricate (possibly overfit) patterns in the training data rather than a simple, clean separation.

(c) Based on this evidence, which kernel would you carry forward for hyperparameter tuning in Part 4? Justify with numbers.

*Your answer:* I would carry forward RBF for tuning in Part 4 (even though linear had the best F1 here) because RBF had the highest accuracy (0.793) and precision (0.644), and unlike linear, RBF has tunable hyperparameters (C and gamma) that can be optimized via GridSearchCV to potentially improve its recall and F1 further — linear has less room for this kind of tuning.


---

## Part 3 — Visualizing the Margin: What Does C Actually Do?

**Task 3.1 — Objective:** Reduce your scaled features to 2 dimensions and visualize how an RBF-kernel SVM's decision boundary changes as C increases.

**What to do:**
1. Use `PCA(n_components=2)` on your scaled training features. Report the total explained variance ratio of the two components.
2. Train three RBF-kernel SVMs on this 2-D projection using a small, medium and large C (choose 3 values spanning at least two orders of magnitude, e.g. 0.01, 1, 100).
3. For each C, plot the 2-D decision boundary (mesh grid + `contourf`) with training points overlaid, and report the number of support vectors.

**Hint — Building the Mesh Grid:**
- Build a fine grid of (x, y) spanning slightly beyond your projected data's min/max using `np.meshgrid`.
- Call `model.predict()` on every grid point (flatten, then reshape back to grid shape).
- `plt.contourf(xx, yy, Z, alpha=0.3)` draws shaded regions; `plt.scatter(...)` overlays real points colored by true label.


In [ ]:
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())


In [ ]:
C_values = [0.01, 1, 100]
c_models = {}

for C in C_values:
    m = SVC(kernel='rbf', C=C, random_state=42)
    m.fit(X_train_pca, y_train)
    c_models[C] = m
    print(f"C={C}  Support vectors: {m.support_.shape[0]}")


In [ ]:
x_min, x_max = X_train_pca[:,0].min()-1, X_train_pca[:,0].max()+1
y_min, y_max = X_train_pca[:,1].min()-1, X_train_pca[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

fig, axes = plt.subplots(1, 3, figsize=(18,5))
for ax, C in zip(axes, C_values):
    m = c_models[C]
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3)
    ax.scatter(X_train_pca[:,0], X_train_pca[:,1], c=y_train, s=10, edgecolor='k')
    ax.set_title(f"C={C}  (SVs={m.support_.shape[0]})")
plt.tight_layout()
plt.show()


**Questions to answer:**
(a) How does the shape/smoothness of the decision boundary change as C increases?

*Your answer:* At C=0.01, the decision boundary is very smooth and simple, barely bending to fit the data. As C increases to 1 and then 100, the boundary becomes progressively more complex and wiggly, bending around individual points to reduce training error.

(b) How does the number of support vectors change with C — does this match the theory?

*Your answer:* The number of support vectors was 2995 at C=0.01, 2658 at C=1, and 2641 at C=100 — decreasing as C increases. This matches theory: a small C allows a wider margin (more points become support vectors, more misclassification tolerated), while a large C forces a narrower margin that only needs the "hardest" boundary-defining points as support vectors.

(c) Which C looks like it is overfitting on this 2-D view, and which looks like it is underfitting? Explain the visual evidence.

*Your answer:* C=100 shows signs of overfitting — the boundary bends tightly around small clusters of points, chasing individual training examples rather than a general trend. C=0.01 shows underfitting — the boundary is almost a straight/smooth curve that ignores a lot of local structure in the data, trading training accuracy for simplicity.

(d) Why is this 2-D plot only an APPROXIMATE illustration of the real (high-dimensional) SVM? What could it be misleading you about?

*Your answer:* This plot only uses 2 PCA components, which together explain about 45% of the total variance (0.332 + 0.120) — meaning more than half the information in the original 30 features is lost. The real SVM makes its decisions using all 30 dimensions, so the boundary shown here is a simplified shadow of the true, much higher-dimensional boundary; it could be misleading about how "complex" or "simple" the real decision surface actually is, since relationships visible or invisible in 2D may look completely different in the full feature space.


---

## Part 4 — Systematic Hyperparameter Search

**Task 4.1 — Objective:** Run `GridSearchCV` over a C x gamma grid for an RBF-kernel SVM and select the best configuration using cross-validated F1-score.

**What to do:**
1. Define a parameter grid with at least 3 values of C and at least 3 values of gamma.
2. Run `GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='f1')` on your scaled **training** data.
3. Extract `gs.cv_results_` and reshape the mean test scores into a C x gamma grid.
4. Visualize as a heatmap (`imshow` or a pivoted DataFrame) with C and gamma labeled.
5. Refit the best estimator and report its performance on the held-out **test** set.

| Syntax | Description |
|---|---|
| `GridSearchCV(estimator, param_grid, cv, scoring)` | tries every combination via k-fold CV |
| `gs.best_params_` | best hyperparameter combination |
| `gs.cv_results_['mean_test_score']` | mean CV score per combination |


In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.001, 0.01, 0.1]
}

gs = GridSearchCV(SVC(kernel='rbf', random_state=42), param_grid, cv=5, scoring='f1', n_jobs=-1)
gs.fit(X_train_s, y_train)

print("Best params:", gs.best_params_)
print("Best CV F1-score:", gs.best_score_)


In [ ]:
scores = gs.cv_results_['mean_test_score'].reshape(len(param_grid['C']), len(param_grid['gamma']))
score_df = pd.DataFrame(scores, index=param_grid['C'], columns=param_grid['gamma'])

plt.figure(figsize=(6,5))
plt.imshow(scores, cmap='viridis', aspect='auto')
plt.xticks(range(len(param_grid['gamma'])), param_grid['gamma'])
plt.yticks(range(len(param_grid['C'])), param_grid['C'])
plt.xlabel('gamma')
plt.ylabel('C')
plt.colorbar(label='Mean CV F1-score')
plt.title('GridSearchCV: F1-score over C x gamma')
for i in range(len(param_grid['C'])):
    for j in range(len(param_grid['gamma'])):
        plt.text(j, i, f"{scores[i,j]:.3f}", ha='center', va='center', color='white')
plt.show()
score_df


In [ ]:
best_svm = gs.best_estimator_
y_pred_best = best_svm.predict(X_test_s)

acc_best = accuracy_score(y_test, y_pred_best)
prec_best = precision_score(y_test, y_pred_best)
rec_best = recall_score(y_test, y_pred_best)
f1_best = f1_score(y_test, y_pred_best)
print("TUNED RBF on TEST SET — Accuracy:", acc_best, "Precision:", prec_best, "Recall:", rec_best, "F1:", f1_best)


**Questions to answer:**
(a) What were the best C and gamma values found, and the corresponding cross-validated F1-score?

*Your answer:* The best parameters found were C=10 and gamma=0.01, with a cross-validated F1-score of 0.5860.

(b) Is there a clear "best region" in the heatmap, or do several combinations perform similarly? What does that tell you?

*Your answer:* There isn't one sharply dominant cell — C=10/gamma=0.01 (0.586) is very close to C=10/gamma=0.001 (0.584) and C=1/gamma=0.1 (0.573). There's a clear "bad" region though: C=0.1 with gamma=0.001 scored 0.0 (essentially predicting one class only), showing that an overly weak model with a too-small gamma fails completely. The fact that several nearby combinations score similarly suggests the model's performance is reasonably stable across that region of the grid, rather than being extremely sensitive to one exact setting.

(c) Did the cross-validated score and the final test-set score agree closely, or was there a gap? What would a large gap suggest?

*Your answer:* They agreed closely — cross-validated F1 was 0.586 and the test-set F1 was 0.557, a small gap of about 0.03. A large gap would suggest the model was overfitting to the cross-validation folds (performing well on data it indirectly "saw" during tuning but poorly on truly unseen data); the small gap here suggests the tuned model generalizes reasonably well.

(d) Compare the tuned RBF model's test performance to the un-tuned RBF model from Part 2. Was tuning worth it?

*Your answer:* The un-tuned RBF (Part 2) had Accuracy 0.793, Precision 0.644, Recall 0.489, F1 0.556. The tuned RBF had Accuracy 0.789, Precision 0.630, Recall 0.500, F1 0.557. The results are nearly identical — accuracy and precision actually dropped slightly, while recall and F1 improved marginally. In this case, tuning was barely worth the extra computational cost; the default RBF parameters were already close to the optimum for this dataset.


---

## Part 5 — Class Imbalance and the Business Trade-off

**Task 5.1 — Objective:** Compare your best RBF-kernel SVM with and without `class_weight='balanced'`, and reason about which is more useful for the business.

**What to do:**
1. Using your best (C, gamma) from Part 4, train two RBF SVMs: `class_weight=None` and `class_weight='balanced'`.
2. Report Precision, Recall, F1-score and confusion matrix for both on the test set.
3. Plot an ROC curve or Precision-Recall curve for the balanced model.


In [ ]:
best_C = gs.best_params_['C']
best_gamma = gs.best_params_['gamma']

svm_none = SVC(kernel='rbf', C=best_C, gamma=best_gamma, class_weight=None, probability=True, random_state=42)
svm_none.fit(X_train_s, y_train)
pred_none = svm_none.predict(X_test_s)

svm_bal = SVC(kernel='rbf', C=best_C, gamma=best_gamma, class_weight='balanced', probability=True, random_state=42)
svm_bal.fit(X_train_s, y_train)
pred_bal = svm_bal.predict(X_test_s)

for name, pred in [('class_weight=None', pred_none), ('class_weight=balanced', pred_bal)]:
    print(name)
    print(" Precision:", precision_score(y_test, pred))
    print(" Recall:", recall_score(y_test, pred))
    print(" F1:", f1_score(y_test, pred))
    cm = confusion_matrix(y_test, pred)
    print(" Confusion matrix:\n", cm)
    print()


In [ ]:
probs_bal = svm_bal.predict_proba(X_test_s)[:,1]

fpr, tpr, _ = roc_curve(y_test, probs_bal)
roc_auc = auc(fpr, tpr)

prec_curve, rec_curve, _ = precision_recall_curve(y_test, probs_bal)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
axes[0].plot([0,1],[0,1],'--', color='gray')
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — balanced SVM")
axes[0].legend()

axes[1].plot(rec_curve, prec_curve)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — balanced SVM")
plt.tight_layout()
plt.show()


**Questions to answer:**
(a) Precisely how did precision and recall move when you switched to `class_weight='balanced'`? Give the numbers.

*Your answer:* With `class_weight=None`: Precision 0.630, Recall 0.500. With `class_weight='balanced'`: Precision 0.506, Recall 0.775. Precision dropped by about 0.124 while recall jumped by about 0.275 — the balanced model trades a significant amount of precision for a much larger gain in recall.

(b) Is it more costly to miss an actual churner (false negative) or wrongly flag a loyal customer (false positive)? Which model would you deploy?

*Your answer:* Missing an actual churner (false negative) is more costly for the business — once that customer leaves, the opportunity to retain them (with a discount, outreach, etc.) is gone, while a false positive just means offering a retention incentive to someone who wasn't going to leave anyway, a smaller waste of resources. Given this, I'd deploy the `class_weight='balanced'` model, since it catches far more actual churners (Recall 0.775 vs 0.500), even though it raises more false alarms.

(c) SVM does not directly output a "probability of churn" like Logistic Regression. What did you have to enable in `SVC()` to get probability-like scores?

*Your answer:* I had to set `probability=True` when creating the `SVC()` model. This enables Platt scaling internally, which calibrates the SVM's decision function output into probability-like scores accessible via `.predict_proba()`.


---

## Part 6 — Model Showdown: SVM vs. Everything Else

**Task 6.1 — Objective:** Build a single comparison table across all five algorithms (Decision Tree, Random Forest, Logistic Regression, KNN, SVM) and select ONE final model for production.

**What to do:**
1. Build a table: one row per model, columns Accuracy / Precision / Recall / F1-score. Re-run earlier models here on the same split if you don't have saved numbers.
2. Produce ONE bar chart comparing F1-score across all five models.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)

knn = KNeighborsClassifier(n_neighbors=21)
knn.fit(X_train_s, y_train)
pred_knn = knn.predict(X_test_s)

all_models = {
    'Decision Tree': pred_dt,
    'Random Forest': pred_rf,
    'Logistic Regression': pred_lr,
    'KNN': pred_knn,
    'SVM (tuned RBF, balanced)': pred_bal
}

rows = []
for name, pred in all_models.items():
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred)
    })

model_comparison = pd.DataFrame(rows)
model_comparison


In [ ]:
plt.figure(figsize=(8,5))
plt.bar(model_comparison['Model'], model_comparison['F1-score'])
plt.ylabel("F1-score")
plt.title("F1-score Comparison Across All Models")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


**Questions to answer:**
(a) Which model wins on F1-score? Which wins on recall? Are they the same model?

*Your answer:* The balanced SVM wins on F1-score (0.6125), very narrowly ahead of Logistic Regression (0.6092). The balanced SVM also wins on recall by a wide margin (0.775 vs Logistic Regression's 0.567). So yes, in this case the same model (balanced SVM) wins on both F1 and recall.

(b) SVM is generally slower to train than Decision Tree/Random Forest here. Was the extra cost worth it?

*Your answer:* Given that the tuned/balanced SVM narrowly beat Logistic Regression on F1 and meaningfully beat every other model on recall (which matters most for churn, as discussed in 5(b)), the extra training cost was worth it — especially since, once scaled, SVM training itself was fast (under 1 second); the real cost was in the GridSearchCV hyperparameter search, which is a one-time cost during model development, not something repeated at prediction time.

(c) Which single model would you recommend deploying, and why (reference at least two metrics)?

*Your answer:* I would recommend the balanced-class-weight SVM for deployment. It has the highest recall (0.775) of all five models by a wide margin, meaning it catches the most actual churners — which matters most given that missing a churner is more costly than a false alarm (as established in Part 5). It also has the best F1-score (0.6125) overall, showing this isn't a one-sided trade-off — it still balances precision and recall better than any other model on net.


---

## Part 7 — Final Predictions and Wrap-Up

**Task 7.1 — Objective:** Generate a predictions file and write a short executive summary.

**What to do:**
1. Using your final chosen model, generate predictions on the test set and save `svm_predictions.csv` with true label, predicted label, and predicted churn probability/score.
2. Write a 5-8 sentence Executive Summary below: which model you chose, key metrics, the main trade-off you made in Part 5, and one limitation of your analysis.


In [ ]:
final_probs = svm_bal.predict_proba(X_test_s)[:,1]
final_preds = svm_bal.predict(X_test_s)

predictions_df = pd.DataFrame({
    'True_Label': y_test.values,
    'Predicted_Label': final_preds,
    'Predicted_Churn_Score': final_probs
})
predictions_df.to_csv('svm_predictions.csv', index=False)
predictions_df.head()


**Executive Summary**

For this churn prediction task, I recommend deploying the RBF-kernel SVM with `class_weight='balanced'` (C=10, gamma=0.01) as the final production model. On the test set, it achieves Accuracy 0.740, Precision 0.506, Recall 0.775, and F1-score 0.6125 — the highest F1 and by far the highest recall among all five models tested (Decision Tree, Random Forest, Logistic Regression, KNN, and SVM). The central trade-off, explored in Part 5, was precision vs. recall: switching from `class_weight=None` to `class_weight='balanced'` dropped precision from 0.630 to 0.506 but raised recall from 0.500 to 0.775. I judged this trade-off worthwhile because, in a churn context, failing to catch an actual churner (false negative) is more costly to the business than wasting a retention offer on a loyal customer (false positive). One key limitation of this analysis is that it optimizes purely for F1/recall on a fixed test split rather than directly for business cost (e.g. the actual dollar cost of a false negative vs a false positive), and the hyperparameter search in Part 4 covered only a modest 3x3 grid, so a finer or wider search might still find a better-performing configuration.

**Final Reflection**
(a) What was the single most surprising result in this lab?

*Your answer:* The most surprising result was how dramatically scaling affected training time — the unscaled linear SVM took about 376 seconds to train versus under 1 second once scaled, while barely changing accuracy. I expected scaling to help, but not by that many orders of magnitude.

(b) If you had one more hour, what would you try next?

*Your answer:* I would widen the GridSearchCV grid (more C and gamma values, especially around C=10, gamma=0.01 where the best score was found) to see if a finer search finds an even better combination, and I would also try adjusting the classification threshold on the balanced SVM's probability scores (instead of just using the default 0.5 cutoff) to see if there's a better precision/recall trade-off point for this specific business case.
